# 03. 분기와 병렬 실행 (Branching & Parallel Execution)

> LangGraph의 superstep 모델은 노드 병렬 실행을 자연스럽게 지원해요. Fan-out/Fan-in과 리듀서, 동적 분기를 묶어 병렬 워크로드의 정석을 배워요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. Fan-out / Fan-in 패턴으로 노드를 병렬 실행하고 결과를 하나로 모을 수 있어요
2. `Annotated[list, operator.add]` 리듀서(Reducer)로 여러 노드의 결과를 안전하게 병합할 수 있어요
3. `add_conditional_edges()`로 런타임 상태에 따른 동적 분기를 구성할 수 있어요
4. 커스텀 리듀서(Custom Reducer)로 병렬 결과를 신뢰도 기반으로 정렬·집계할 수 있어요
5. Superstep 트랜잭션 특성을 이해하고 병렬 실행 중 예외를 올바르게 처리할 수 있어요

## 사전 지식

- `02-Subgraphs.ipynb` — 서브그래프 구성, 부모-자식 상태 변환
- LangGraph 기초: `StateGraph`, `add_node`, `add_edge`, `compile`, `invoke`
- Python `TypedDict`, `Annotated` 타입 힌트 기본 이해


## 핵심 개념: Fan-out / Fan-in

**Fan-out**은 하나의 노드에서 여러 노드로 동시에 분기하는 패턴이에요. **Fan-in**은 반대로 여러 노드의 결과를 하나의 노드로 모으는 패턴이에요.

피자를 만드는 과정을 예로 들어볼게요:
- Fan-out: 도우 준비, 소스 준비, 치즈 준비를 **동시에** 시작해요
- Fan-in: 준비된 재료를 모두 **합쳐서** 완성 피자를 만들어요

이 패턴을 LangGraph에서 구현하면 **전체 처리 시간을 크게 단축**할 수 있어요!

```mermaid
flowchart LR
    S([시작<br>START]) --> A[노드 A<br>Fan-out 시작]
    A --> B[노드 B<br>병렬 실행]
    A --> C[노드 C<br>병렬 실행]
    B --> D[노드 D<br>Fan-in 집결]
    C --> D
    D --> E([끝<br>END])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef parallel fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class S input
    class A process
    class B,C parallel
    class D,E output
```

### 핵심 구성 요소

| 구성 요소 | 역할 | 예시 |
|-----------|------|------|
| `add_edge("a", "b")` + `add_edge("a", "c")` | Fan-out: 하나에서 여러 개로 분기 | A → B, A → C 동시 실행 |
| `add_edge(["b", "c"], "d")` | Fan-in: 여러 개를 하나로 모음 | B, C 완료 후 D 실행 |
| `Annotated[list, operator.add]` | 병렬 결과를 안전하게 누적 | 각 노드의 리스트를 합쳐요 |
| `add_conditional_edges()` | 런타임 상태에 따른 동적 분기 | 상태 값으로 경로 선택 |
| `defer=True` | 모든 선행 작업 완료 대기 | 명시적 의존성 없이 동기화 |

> 🔑 **핵심 개념**: `operator.add` 리듀서는 여러 병렬 노드가 같은 상태 키에 쓸 때 값을 덮어쓰지 않고 **리스트를 연결(concatenate)**해요. 병렬 실행에서 상태 충돌을 방지하는 핵심 메커니즘이에요.


## 환경 설정


In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어요)
from dotenv import load_dotenv

load_dotenv(override=True)


In [ ]:
import os

# LangSmith 추적 설정 (실행 흐름을 시각화해서 확인할 수 있어요)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Branching-Parallel"


## 1. 기본 Fan-out / Fan-in

가장 단순한 형태의 병렬 실행 패턴을 살펴볼게요. 노드 A에서 B와 C로 팬아웃하고, B와 C의 결과를 D에서 팬인하는 구조예요.

### 왜 리듀서(Reducer)가 필요한가요?

병렬 노드 B와 C가 같은 상태 키 `aggregate`에 동시에 값을 쓰면 어떻게 될까요? 기본 동작은 **마지막으로 쓴 값이 이전 값을 덮어씁니다**. 이건 우리가 원하는 동작이 아니에요!

마치 두 사람이 같은 문서에 동시에 쓰면 한 사람의 내용이 사라지는 것과 같아요. `Annotated[list, operator.add]`를 사용하면 각 노드의 출력이 **기존 리스트에 누적**되어 모든 결과를 보존할 수 있어요.

| reducer 없을 때 | `operator.add` reducer 사용 |
|:---|:---|
| B가 `["B"]` 쓰고 C가 `["C"]` 쓰면 | B가 `["B"]` 쓰고 C가 `["C"]` 쓰면 |
| 최종: `["C"]` (B 결과 유실!) | 최종: `["B", "C"]` (모두 보존) |


In [ ]:
import operator
from typing import Annotated, Any
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 상태 정의: operator.add 리듀서로 리스트 누적
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → a → (b, c 병렬) → d → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 실행: 빈 리스트에서 시작해서 각 노드가 값을 누적해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


실행 결과를 보면 B와 C가 병렬로 실행될 때 둘 다 `["I'm A"]` 상태를 보고 있어요. 이건 B와 C가 **같은 superstep**에서 동시에 실행되기 때문이에요.

D의 입력을 보면 B와 C의 결과가 모두 누적되어 있어요. `operator.add` 리듀서 덕분에 어느 쪽도 덮어쓰이지 않았어요!


## 2. 다단계 경로의 Fan-out / Fan-in

실제 애플리케이션에서는 하나의 병렬 경로가 여러 단계로 구성될 수 있어요. 예를 들어 `B1 → B2` 경로와 `C` 경로가 병렬로 실행된 후 `D`에서 합쳐지는 구조예요.

이런 경우 `add_edge(["b2", "c"], "d")`처럼 **리스트로 선행 노드를 명시**하면 돼요.


In [ ]:
import operator
from typing import Annotated, Any
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 상태 및 노드 클래스 정의 (이전 섹션과 동일)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → a → (b1 → b2 || c) → d → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 실행 - C가 먼저 끝나도 B2를 기다린 후 D가 실행돼요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. defer=True로 간편한 동기화

그래프가 복잡해지면 `add_edge(["b2", "c"], "d")`처럼 모든 선행 노드를 일일이 나열하기 어려울 수 있어요. LangGraph는 이 문제를 `defer=True` 옵션으로 해결해요.

`add_node("d", ..., defer=True)`로 설정하면 노드 D는 **현재 진행 중인 모든 대기 작업이 완료될 때까지** 자동으로 기다려요.

### 명시적 Fan-in vs defer=True

| 방법 | 코드 | 유연성 | 적합한 상황 |
|------|------|--------|------------|
| 명시적 Fan-in | `add_edge(["b2", "c"], "d")` | 특정 노드만 기다려요 | 어떤 노드를 기다릴지 명확할 때 |
| `defer=True` | `add_node("d", ..., defer=True)` | 모든 대기 작업을 자동으로 기다려요 | 조건부 분기로 실행 노드가 유동적일 때 |


In [ ]:
import operator
from typing import Annotated, Any
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: defer=True 활용: 명시적 fan-in 없이 동기화
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → a → (b1 → b2 || c) → d(defer) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: C가 먼저 끝나도 defer=True 덕분에 B2를 기다린 후 D가 실행돼요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. Superstep 트랜잭션과 예외 처리

### Superstep이란?

LangGraph의 병렬 실행은 **Superstep** 단위로 처리돼요. Superstep이란 동시에 실행되는 노드들의 묶음이에요. 은행 계좌이체를 생각해보세요. 출금과 입금이 **둘 다 성공하거나 둘 다 취소**되어야 하죠. Superstep도 마찬가지예요.

중요한 특성: Superstep은 **트랜잭션(Transaction)**처럼 동작해요.
- 한 Superstep 내 **어느 하나라도 실패**하면 전체 Superstep이 롤백돼요
- 상태 업데이트가 **전혀 적용되지 않아요**

```mermaid
flowchart LR
    A[노드 A] --> B[노드 B<br/>성공]
    A --> C[노드 C<br/>실패!]
    B --> D[노드 D]
    C --> D

    subgraph SS[Superstep - 트랜잭션]
        B
        C
    end

    style C fill:#f8d7da,stroke:#dc3545,color:#721c24
    style SS fill:#fff3cd,stroke:#ffc107
```


In [ ]:
import operator
from typing import Annotated, Any
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Superstep 트랜잭션 동작 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 방법 1: 노드 내 try/except로 예외 처리
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 조건부 분기 (Conditional Branching)

Fan-out 경로가 고정적이지 않고 **런타임 상태에 따라 달라져야** 할 때 `add_conditional_edges()`를 사용해요.

라우팅 함수가 `Sequence[str]`(문자열 리스트)를 반환하면 여러 노드로 동시에 분기할 수 있어요.

### `path_map`와 공통 합류 노드

LangGraph v1.x 기준 `add_conditional_edges()`의 형태는 다음과 같아요.

```python
builder.add_conditional_edges(source, path, path_map=None)
```

- `source`: 분기를 시작할 노드
- `path`: 현재 상태를 보고 다음 노드 이름 하나 또는 여러 개를 반환하는 라우팅 함수
- `path_map`: 라우팅 함수가 보낼 수 있는 후보 노드 목록/매핑. 후보를 명시하면 그래프 구조 검증과 시각화가 정확해져요.

중요한 점은 `path_map`이 **갈 수 있는 노드 후보를 알려주는 역할**만 한다는 거예요. 후보 노드들 뒤에 공통 합류 노드까지 자동으로 연결하지는 않아요.

### 공통 노드 `e`로 합류시키는 차이

| 구성 | 어떤 노드가 실행되나 | 결과 |
|------|----------------------|------|
| `add_conditional_edges("a", route, ["b", "c", "d"])`만 사용 | `a` 다음에 선택된 `b,c` 또는 `c,d`만 실행 | `e`는 실행되지 않음 |
| 위 코드 + `for node in ...: add_edge(node, "e")` | 선택된 분기들이 끝난 뒤 `e`가 한 번 실행 | `e`가 공통 fan-in/sink 역할 |

> 💡 **핵심**: `path_map`은 조건부 분기가 갈 수 있는 후보 목록/매핑이고, 공통 합류 노드를 자동으로 만들지는 않아요. 조건부 분기 뒤에 반드시 실행되어야 하는 공통 노드가 있다면 `for node in intermediates: builder.add_edge(node, "e")`처럼 각 후보 노드에서 합류 노드로 가는 일반 엣지를 명시적으로 추가하세요.


In [ ]:
import operator
from typing import Annotated, Any, Sequence
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 조건부 분기: 상태 값에 따라 다른 노드로 fan-out
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → a → {조건부 분기: b,c 또는 c,d} → e → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: which='bc': B와 C로 분기해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: which='cd': C와 D로 분기해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 커스텀 리듀서: 신뢰도 기반 집계

병렬 노드의 결과를 단순히 누적하는 것 이상이 필요할 때 **커스텀 리듀서(Custom Reducer)**를 사용해요.

예를 들어, 여러 분석 모델이 병렬로 실행될 때 각 결과에 **신뢰도(reliability)** 점수가 있다면, 가장 신뢰도 높은 결과부터 정렬해서 최종 결론에 반영하고 싶을 수 있어요.

```mermaid
flowchart TD
    A[노드 A<br>분기 결정] --> B[노드 B<br>reliability=0.1]
    A --> C[노드 C<br>reliability=0.9]
    A --> D[노드 D<br>reliability=0.5]
    B --> E[노드 E<br>신뢰도 순 정렬 집계]
    C --> E
    D --> E
    E --> END([끝])

    subgraph FR[fanout_values에 신뢰도 포함]
        B
        C
        D
    end

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef parallel fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A process
    class B,C,D parallel
    class E,END output
```

> 🔑 **핵심 개념**: 커스텀 리듀서를 사용하면 `fanout_values`라는 별도 필드에 {값, 신뢰도} 형태로 결과를 모은 뒤, 집계 노드에서 정렬·가공할 수 있어요. 이것이 **Map-Reduce 패턴**의 핵심이에요.


In [ ]:
import operator
from typing import Annotated, Any, Sequence
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 커스텀 리듀서: fanout_values 누적 및 초기화 처리
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → a → {조건부: b,c 또는 c,d} → e(집계) → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: which='bc': B(0.1)와 C(0.9)가 실행돼요 - C가 높은 신뢰도라 먼저 집계돼요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: which='cd': C(0.9)와 D(0.5)가 실행돼요 - C가 더 높은 신뢰도
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 최종 실전 예제: LLM 고객지원 티켓 병렬 점검 그래프

앞의 예제는 구조를 보기 쉽게 하려고 작은 문자열과 규칙 기반 라우팅을 사용했어요. 마지막 예제에서는 같은 패턴을 실제 업무에 더 가깝게 바꿔봅니다.

이번 목표는 **LLM이 고객 문의를 읽고 필요한 점검 노드를 여러 개 선택**한 뒤, 선택된 점검 노드들이 병렬로 실행되고, 마지막에 LLM이 답변 초안을 작성하는 그래프를 만드는 거예요.

```mermaid
flowchart TD
    A[고객 문의] --> B[LLM analyze_ticket]
    B --> C{selected_checks}
    C -->|환불/결제| D[refund_policy_check]
    C -->|로그인/오류| E[technical_check]
    C -->|프리미엄/긴급| F[sla_priority_check]
    C -->|일반 문의| G[general_support_check]
    D --> H[LLM draft_customer_reply]
    E --> H
    F --> H
    G --> H
    H --> I[최종 답변]
```


### 실행 방법

아래 셀들을 순서대로 실행하세요.

실행 전 `.env`에 `OPENAI_API_KEY`가 준비되어 있어야 합니다. 이 예제는 두 곳에서 LLM을 호출합니다.

1. `analyze_ticket`: 고객 문의를 읽고 병렬로 실행할 점검 노드 목록을 구조화 출력으로 선택합니다.
2. `draft_customer_reply`: 병렬 점검 결과를 바탕으로 고객에게 보낼 답변 초안을 작성합니다.

샘플 실행 후 마지막 셀의 `my_ticket`과 `my_customer_tier`만 바꿔서 직접 테스트할 수 있어요.


### 1단계 — State와 reducer 정의하기

병렬 점검 노드들은 모두 `findings`와 `trace`를 업데이트합니다. 여러 노드가 같은 키를 동시에 업데이트하므로 reducer가 필요해요.

- `collect_findings`: 병렬 점검 결과 리스트를 누적합니다.
- `operator.add`: 실행 흐름 로그(`trace`)를 리스트로 이어 붙입니다.


In [ ]:
import operator
from typing import Annotated, Any, Literal, Sequence
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langgraph.graph import START, END, StateGraph

load_dotenv(".env", override=True)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 1단계: import + State/reducer 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 2단계 — LLM 라우터 만들기

`analyze_ticket` 노드는 LLM 구조화 출력으로 어떤 점검 노드를 실행할지 결정합니다.

예를 들어 “프리미엄 고객인데 결제가 두 번 됐고 빨리 처리해 주세요” 같은 문의는 `refund_policy_check`와 `sla_priority_check`를 함께 선택할 수 있어요. 이렇게 라우팅 결과가 리스트가 되면 다음 단계에서 조건부 fan-out이 발생합니다.


In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

router_model = init_chat_model("openai:gpt-4o-mini", temperature=0)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 2단계: LLM 구조화 출력으로 병렬 점검 계획 세우기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3단계 — 병렬 점검 노드 만들기

각 점검 노드는 서로 독립적으로 실행됩니다. 그래서 같은 superstep에서 병렬로 실행될 수 있고, 결과는 `findings` reducer로 합쳐집니다.

각 finding에는 다음 정보를 담습니다.

- `area`: 어떤 영역의 점검인지
- `priority`: 답변에서 얼마나 우선해야 하는지
- `confidence`: 점검 결과 신뢰도
- `summary`: 고객에게 설명할 핵심 내용
- `action`: 다음 조치


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3단계: 병렬 점검 노드 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 4단계 — LLM 답변 작성 노드와 그래프 조립하기

`draft_customer_reply`는 병렬 점검 결과를 우선순위와 신뢰도 기준으로 정렬한 뒤, LLM에게 고객 답변 초안을 작성하게 합니다.

그래프 조립에서 중요한 부분은 두 가지예요.

1. `add_conditional_edges("analyze_ticket", route_to_checks, CHECK_NODE_NAMES)`로 LLM이 선택한 점검 노드들로 fan-out합니다.
2. 각 점검 노드에서 `draft_customer_reply`로 일반 엣지를 추가해 공통 fan-in 지점을 명시합니다.


In [ ]:
reply_model = init_chat_model("openai:gpt-4o-mini", temperature=0)

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 4단계: LLM 답변 작성 노드 + 그래프 조립
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 5단계 — 그래프 구조 확인하기

그래프를 시각화하면 `analyze_ticket`에서 여러 점검 노드로 뻗어 나간 뒤, 다시 `draft_customer_reply`로 모이는 구조를 확인할 수 있습니다.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: LLM 라우팅 → 조건부 병렬 점검 → LLM 답변 작성 → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 6단계 — 샘플 티켓으로 실행 흐름 추적하기

`run_ticket_case()`는 한 건의 티켓을 실행하고, LLM이 선택한 병렬 점검 목록과 최종 답변을 함께 출력합니다.

샘플을 보면 입력 문장에 따라 실행되는 점검 노드 조합이 달라지는 것을 확인할 수 있어요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 6단계: 실행 도우미 함수 + 샘플 티켓 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 7단계 — 직접 티켓을 바꿔 실행하기

마지막으로 `my_ticket`과 `my_customer_tier`만 바꿔서 실행해 보세요. LLM 라우터가 어떤 병렬 점검 노드를 선택하는지 확인하는 것이 핵심입니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 7단계: 직접 티켓을 바꿔 실행하기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Fan-out / Fan-in**: `add_edge("a", "b")` + `add_edge("a", "c")`로 분기하고, `add_edge(["b", "c"], "d")`로 합쳐요
- **Annotated[list, operator.add]**: 병렬 노드가 같은 키에 쓸 때 덮어쓰지 않고 리스트를 안전하게 누적해요
- **ReturnNodeValue 클래스 패턴**: 노드 로직을 클래스로 캡슐화해 서로 다른 값을 주입할 수 있어요
- **defer=True**: 모든 선행 작업을 자동으로 기다려 복잡한 의존성을 간단하게 처리해요
- **Superstep 트랜잭션**: 병렬 분기 중 하나라도 실패하면 전체 superstep이 롤백돼요
- **add_conditional_edges()**: 라우팅 함수가 `Sequence[str]`을 반환해 런타임에 동적으로 분기해요
- **path_map/가능한 분기 목록**: `add_conditional_edges("a", route, ["b", "c", "d"])`처럼 후보를 명시하면 구조가 명확하고 시각화가 정확해져요
- **공통 합류 노드**: 조건부 분기 뒤의 sink 노드는 `for node in intermediates: add_edge(node, "e")`처럼 명시적으로 연결해요
- **커스텀 리듀서**: `reduce_fanouts` 같은 함수로 복잡한 병합 로직을 구현할 수 있어요
- **최종 실전 예제**: LLM이 고객지원 티켓을 읽고 필요한 병렬 점검 노드를 선택한 뒤, reducer로 결과를 모아 LLM 답변 초안을 작성하는 흐름을 구현했어요

### 패턴 비교표

| 패턴 | 사용 시점 | 핵심 API |
|------|-----------|----------|
| 기본 Fan-out/Fan-in | 경로가 미리 정해진 병렬 처리 | `add_edge` |
| 다단계 경로 | 병렬 경로 내에 순차 단계 존재 | `add_edge([list], node)` |
| defer=True | 복잡한 의존성, map-reduce | `add_node(..., defer=True)` |
| 조건부 분기 | 런타임 상태에 따른 동적 분기 | `add_conditional_edges` |
| 조건부 분기 + 공통 합류 | 동적 분기 후 반드시 공통 집계 노드 실행 | `add_conditional_edges` + `add_edge` |
| LLM 라우팅 + 병렬 점검 | 자연어 입력에서 여러 점검 노드를 동적으로 선택 | `init_chat_model` + `add_conditional_edges` |
| 커스텀 리듀서 | 결과 정렬, 가중치 집계 | `Annotated[list, custom_fn]` |


## 다음 노트북 예고

다음 `04-Durable-Execution.ipynb`에서는 **내구성 실행(Durable Execution)**을 배워요. 체크포인터를 활용해 그래프 실행 도중 장애가 발생해도 중단 지점부터 재개하는 방법, 그리고 retry 정책으로 실패한 노드를 자동으로 재시도하는 방법을 살펴볼 거예요. 오늘 배운 병렬 실행 패턴에 내구성을 더하면 프로덕션에 배포할 수 있는 견고한 그래프를 만들 수 있어요!
